## 0. Setup (run first, every session) - and how your progress is protected
**Colab:** checkpoints/results are written *directly to your Google Drive* (symlinks), so a power or internet cut loses at most one checkpoint interval (~300 updates). After reconnecting: re-run section 0, then re-run the same training cell - it resumes automatically. Drive's free quota is 15 GB: one full-FT run is ~12 GB, so use the cleanup cell after each full-FT run.

**Kaggle:** an interactive session's disk is wiped when it ends. Protect yourself with **Save Version -> Save & Run All (Commit)**: it runs on Kaggle's servers even if your internet/power dies, and the output (checkpoints + results, up to 20 GB) is kept. To continue later: in the new notebook use *Add Input -> Notebook Output Files* and pick the previous version; section 0 copies its checkpoints back and training resumes. Do one method per commit run (must finish within the 9 h limit). Settings: Accelerator = GPU, Internet = On.

4. **Not yet run end to end** - these cells follow the same pattern as Task 1 but Task 2 code has only been syntax-checked. Run the smoke steps (2.2) first and expect to debug.
5. COCO 2014 is large (~19 GB of images). On Kaggle, attach a COCO-2014 dataset via *Add Input* and symlink it (cell 2.0); on Colab use `scripts.download_assets --coco`. VQAv2 files are small.

In [ ]:
import os, sys, subprocess, shutil, glob
REPO_URL = "https://github.com/ZainFarhan-Work/ATML.git"
BRANCH   = "main"
ON_KAGGLE = os.path.exists("/kaggle")
ROOT = "/kaggle/working" if ON_KAGGLE else "/content"

if not os.path.exists(f"{ROOT}/ATML"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, f"{ROOT}/ATML"], check=True)
else:
    subprocess.run(["git", "-C", f"{ROOT}/ATML", "pull"], check=False)
os.chdir(f"{ROOT}/ATML/PA 3")
print(os.getcwd())

PERSIST = ["checkpoints", "results", "cached", "splits"]
if not ON_KAGGLE:
    # Colab: keep run state on Drive so it survives disconnects
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/atml_pa3"
    for d in PERSIST:
        os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
        if os.path.isdir(d) and not os.path.islink(d):
            shutil.copytree(d, f"{DRIVE}/{d}", dirs_exist_ok=True); shutil.rmtree(d)
        if not os.path.islink(d):
            os.symlink(f"{DRIVE}/{d}", d)
else:
    # Kaggle: restore from earlier committed runs attached as inputs
    for src_root in glob.glob("/kaggle/input/*/ATML/PA 3") + glob.glob("/kaggle/input/*/*/ATML/PA 3"):
        for d in PERSIST:
            if os.path.isdir(f"{src_root}/{d}"):
                shutil.copytree(f"{src_root}/{d}", d, dirs_exist_ok=True)
                print("restored", d, "from", src_root)

def sync_now():
    # Colab: nothing to do (already on Drive). Kaggle: output is saved when the committed run finishes.
    print("progress is stored in:", "Google Drive" if not ON_KAGGLE else "/kaggle/working (kept by Save & Run All)")

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m scripts.check_environment

## 2.0 Data assets

In [ ]:
# Kaggle: attach COCO-2014 + VQAv2 as inputs, then link them here (adjust the input paths shown in the sidebar)
# !mkdir -p data/coco && ln -sfn /kaggle/input/<coco-dataset>/train2014 data/coco/train2014 && ln -sfn /kaggle/input/<coco-dataset>/val2014 data/coco/val2014
# Colab / anywhere with internet:
!python -m scripts.download_assets --vqa
# !python -m scripts.download_assets --coco

## 2.1 Build splits and cache SigLIP features (Step 1)

In [ ]:
!python -m task2_vlm.build_data
!python -m task2_vlm.cache_features
!python -m task2_vlm.checks
sync_now()

## 2.2 Smoke test (a few updates, then delete)

In [ ]:
!python -m task2_vlm.train --run-name smoke --max-updates 3
!rm -rf results/task2/smoke checkpoints/task2/smoke

## 2.3 Main run: caption alignment -> visual instruction tuning -> natural VQA (4,500 updates, resumable)

In [ ]:
!python -m task2_vlm.train --run-name main --resume
sync_now()

## 2.4 Final evaluation (4 image conditions) and analysis

In [ ]:
!python -m task2_vlm.evaluate --run-name main
!python -m task2_vlm.analyze --run-name main
sync_now()

## 2.5 (Optional) 49-token compression

In [ ]:
!python -m task2_vlm.train --run-name tokens49 --compress --resume
!python -m task2_vlm.evaluate --run-name tokens49 --compress
!python -m task2_vlm.analyze --run-name tokens49
sync_now()